In [123]:
import os
import re
import faiss
import spacy
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

In [124]:
with open('data.txt', 'r') as file:
    data = file.read().lower()
data = re.sub('\d+\.',"", data)
data = re.sub(r"\b\d+(?:\.\d+)*\.?\b", "", data)
data = re.sub('[^0-9a-zA-Z\s]', "", data).strip()
data = re.sub("\s+", " ", data).strip()

In [125]:
nlp = spacy.load('en_core_web_sm')

In [126]:
tokens = nlp(data)
lemmatize_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = " ".join(lemmatize_tokens).strip()

In [127]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=40, 
    separators=["\n\n","\n",".","!"," ",""])
data = text_splitter.split_text(data)

In [128]:
doc_ids = [f"doc_{i}" for i in range(len(data))]

In [129]:
ground_truth = {
    "What is Python?": ["doc_0"],
    "Explain machine learning": ["doc_1", "doc_3"],
    "Explain GenAi": ["doc_22", "doc_25", 'doc_100', 'doc_98'],
    "What is a RAG system?": ["doc_5", "doc_2", "doc_7"],
    "Tell me about NLP": ["doc_4", "doc_6"],
}

In [130]:
embeddings_model = SentenceTransformer(
    model_name_or_path = 'sentence-transformers/all-miniLM-L6-V2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [131]:
embeddings = embeddings_model.encode(data).astype('float32')

In [132]:
dimension = embeddings.shape[1]

In [133]:
faiss.normalize_L2(embeddings)

In [134]:
index_faiss_db = faiss.IndexFlatIP(dimension) # dot product 

In [135]:
index_faiss_db.add(embeddings)

In [136]:
def retrieve(query: str, k: int = 5):
    """Encode query and return top-k matching doc_ids."""
    query_embedding = embeddings_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)

    scores, indices = index_faiss_db.search(query_embedding, k)
    retrieved = [doc_ids[i] for i in indices[0]]
    return retrieved 

In [137]:
def precision_at_k(retrieved: list, relevant: list, k: int) -> float:
    """
    Precision@k = (# relevant docs in top-k) / k
    How many of our top-k results were actually relevant?
    """
    retrieved_k = set(retrieved[:k])
    relevant_set = set(relevant)
    hits = retrieved_k & relevant_set
    return len(hits) / k

In [138]:
def recall_at_k(retrieved: list, relevant: list, k: int) -> float:
    """
    Recall@k = (# relevant docs in top-k) / (# total relevant docs)
    How many of the relevant docs did we actually find?
    """
    retrieved_k = set(retrieved[:k])
    relevant_set = set(relevant)
    hits = retrieved_k & relevant_set
    return len(hits) / len(relevant_set) if relevant_set else 0.0

In [139]:
def reciprocal_rank(retrieved: list, relevant: list) -> float:
    """
    Reciprocal Rank = 1 / (rank of first relevant doc)
    How quickly did we find the FIRST relevant document?
    """
    relevant_set = set(relevant)
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant_set:
            return 1.0 / rank
    return 0.0  # No relevant doc found

In [140]:
def mean_reciprocal_rank(all_rr: list) -> float:
    """MRR = average of reciprocal ranks across all queries."""
    return sum(all_rr) / len(all_rr) if all_rr else 0.0

In [141]:
K = 5  # Evaluate top-3 results

results = []
rr_scores = []

print(f"\n{'='*55}")
print(f"  Retrieval Evaluation  (k = {K})")
print(f"{'='*55}")


  Retrieval Evaluation  (k = 5)


In [142]:
for query, relevant_docs in ground_truth.items():
    retrieved_docs = retrieve(query, k=K)

    rec = recall_at_k(retrieved_docs, relevant_docs, K)
    prec = precision_at_k(retrieved_docs, relevant_docs, K)
    rr = reciprocal_rank(retrieved_docs, relevant_docs)
    rr_scores.append(rr)

    results.append({
        "query": query,
        "retrieved": retrieved_docs,
        "relevant": relevant_docs,
        "recall": rec,
        "precision": prec,
        "rr": rr
    })

    print(f"\nQuery   : {query}")
    print(f"Relevant: {relevant_docs}")
    print(f"Retrieved (top-{K}): {retrieved_docs}")
    print(f"Recall@{K}   = {rec:.2f}")
    print(f"Precision@{K} = {prec:.2f}")
    print(f"Recip. Rank  = {rr:.2f}")

# ─────────────────────────────────────────────
# STEP 8: Print Aggregate Metrics
# ─────────────────────────────────────────────
avg_recall = sum(r["recall"] for r in results) / len(results)
avg_prec = sum(r["precision"] for r in results) / len(results)
mrr = mean_reciprocal_rank(rr_scores)

print(f"\n{'='*55}")
print(f"  AGGREGATE RESULTS")
print(f"{'='*55}")
print(f"  Mean Recall@{K}    : {avg_recall:.3f}")
print(f"  Mean Precision@{K} : {avg_prec:.3f}")
print(f"  MRR              : {mrr:.3f}")
print(f"{'='*55}")


Query   : What is Python?
Relevant: ['doc_0']
Retrieved (top-5): ['doc_87', 'doc_88', 'doc_32', 'doc_89', 'doc_9']
Recall@5   = 0.00
Precision@5 = 0.00
Recip. Rank  = 0.00

Query   : Explain machine learning
Relevant: ['doc_1', 'doc_3']
Retrieved (top-5): ['doc_1', 'doc_44', 'doc_0', 'doc_92', 'doc_41']
Recall@5   = 0.50
Precision@5 = 0.20
Recip. Rank  = 1.00

Query   : Explain GenAi
Relevant: ['doc_22', 'doc_25', 'doc_100', 'doc_98']
Retrieved (top-5): ['doc_100', 'doc_98', 'doc_22', 'doc_25', 'doc_21']
Recall@5   = 1.00
Precision@5 = 0.80
Recip. Rank  = 1.00

Query   : What is a RAG system?
Relevant: ['doc_5', 'doc_2', 'doc_7']
Retrieved (top-5): ['doc_35', 'doc_27', 'doc_104', 'doc_79', 'doc_42']
Recall@5   = 0.00
Precision@5 = 0.00
Recip. Rank  = 0.00

Query   : Tell me about NLP
Relevant: ['doc_4', 'doc_6']
Retrieved (top-5): ['doc_11', 'doc_87', 'doc_9', 'doc_67', 'doc_8']
Recall@5   = 0.00
Precision@5 = 0.00
Recip. Rank  = 0.00

  AGGREGATE RESULTS
  Mean Recall@5    : 0.300
  